# MapBiomas México — Colección 1. Cobertura y uso del suelo

Mapas anuales de cobertura y uso del suelo de México, 30 m, licencia CC BY 4.0 (https://mexico.mapbiomas.org).

El proyecto los usa como **filtro de la edafología**: el suelo de cada registro agrícola se resume sobre el área de su municipio con la cobertura de su tipo de cultivo (ver `integracion_agricultura_clima.ipynb`).

| Clase | Nombre | Uso en el proyecto |
|---|---|---|
| 19 | Cultivo anual | máscara `anual` (ciclos PV y OI) |
| 36 | Cultivo perenne | máscara `perenne` (ciclo Perennes) |
| 15 | Pastizal cultivado e inducido | máscara `pastizal` (pastos y zacates) |
| 19 + 36 | — | máscara auxiliar `agricola`, respaldo de los perennes |
| 21 | Mosaico de usos | no se usa (casi no aparece en la Colección 1) |

Se descarga desde 2002 porque cada año *t* usa la ventana *t−1..t+1*.

In [1]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling

from proyecto_agronomia.config import ANIOS_MAPBIOMAS, MAPBIOMAS_RAW_DIR
from datasets.mapbiomas import (
    CLASES_AGRICOLAS_MAPBIOMAS,
    CLAVE_NACIONAL,
    descargar_mapbiomas,
    listar_rasters_mapbiomas,
)

2026-11-24 15:27:06.837 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## Descarga programática

El almacenamiento público de MapBiomas (`storage.googleapis.com/mapbiomas-public`) responde 403. Por eso se usa la API de la plataforma web (`prd.plataforma.mapbiomas.org`), que no pide sesión:

1. `POST /maps/exports` solicita la exportación de la cobertura de México para un año;
2. se consulta `GET /maps/exports/{id}` hasta que el estado sea `READY`;
3. se pide la URL firmada (vence en minutos) y se baja el GeoTIFF.

Se obtiene un TIFF nacional por año (uint8, EPSG:4326, ~30 m, nodata 0, ≈250 MB) en `data/raw/mapbiomas/00/<año>.tif`. La función es idempotente: omite los archivos que ya existen y reanuda las exportaciones pendientes que guarda en `manifiesto.json`. La primera descarga completa tarda varias horas.

La API no está documentada y puede cambiar. Los rasters ya descargados no dependen de ella.

También se puede descargar desde la terminal: `python -m proyecto_agronomia.dataset mapbiomas`.

In [2]:
descargar_mapbiomas(anios=ANIOS_MAPBIOMAS)

rasters = listar_rasters_mapbiomas(MAPBIOMAS_RAW_DIR, anios=ANIOS_MAPBIOMAS)[CLAVE_NACIONAL]
print(f"Años disponibles: {min(rasters)}–{max(rasters)} ({len(rasters)} rasters)")

2026-11-24 15:27:07.153 | INFO     | datasets.mapbiomas:descargar_mapbiomas:253 - MapBiomas: 24 archivos por descargar.


2026-11-24 15:27:33.818 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2002 descargado.


2026-11-24 15:27:54.920 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2003 descargado.


2026-11-24 15:28:17.762 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2004 descargado.


2026-11-24 15:28:40.268 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2005 descargado.


2026-11-24 15:29:04.563 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2006 descargado.


... (15 outputs left out)

2026-11-24 15:36:04.237 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2022 descargado.


2026-11-24 15:36:28.170 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2023 descargado.


2026-11-24 15:36:51.728 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2024 descargado.


2026-11-24 15:37:14.655 | INFO     | datasets.mapbiomas:descargar_mapbiomas:278 - MapBiomas 00/2025 descargado.
2026-11-24 15:37:14.656 | SUCCESS  | datasets.mapbiomas:descargar_mapbiomas:300 - MapBiomas: 1 territorio(s) × 24 años disponibles.
Años disponibles: 2002–2025 (24 rasters)


## Verificación de los rasters

Todos los años deben tener la misma malla (CRS, dimensiones y transformación); el cruce con municipios y suelos depende de ello.

In [3]:
metadatos = []
for anio, ruta in rasters.items():
    with rasterio.open(ruta) as fuente:
        metadatos.append(
            {
                "anio": anio,
                "crs": str(fuente.crs),
                "ancho": fuente.width,
                "alto": fuente.height,
                "resolucion_m": round(fuente.res[0] * 111_320),
                "dtype": fuente.dtypes[0],
                "nodata": fuente.nodata,
                "transform": tuple(fuente.transform)[:6],
                "tamanio_mb": round(ruta.stat().st_size / 1e6),
            }
        )
metadatos = pd.DataFrame(metadatos).set_index("anio")

columnas_malla = ["crs", "ancho", "alto", "transform"]
assert (metadatos[columnas_malla].nunique() == 1).all(), "Los rasters no comparten la malla."
display(metadatos.drop(columns="transform"))

,crs,ancho,alto,resolucion_m,dtype,nodata,tamanio_mb
anio,,,,,,,
2002,EPSG:4326,117501,67519,30,uint8,0.0,247
2003,EPSG:4326,117501,67519,30,uint8,0.0,251
2004,EPSG:4326,117501,67519,30,uint8,0.0,248
2005,EPSG:4326,117501,67519,30,uint8,0.0,247
2006,EPSG:4326,117501,67519,30,uint8,0.0,251
2007,EPSG:4326,117501,67519,30,uint8,0.0,253
2008,EPSG:4326,117501,67519,30,uint8,0.0,252
2009,EPSG:4326,117501,67519,30,uint8,0.0,251
2010,EPSG:4326,117501,67519,30,uint8,0.0,250


## Proporción de las clases agrícolas por año

Una lectura submuestreada (1 de cada 10 píxeles por lado, vecino más cercano) basta para ver la tendencia de cada clase. Es una aproximación en píxeles, no en hectáreas: el cruce de la integración sí pondera por el área real de cada píxel.

In [4]:
FACTOR_SUBMUESTREO = 10
CLASES_REPORTE = {**CLASES_AGRICOLAS_MAPBIOMAS, 21: "Mosaico de usos"}

proporciones = {}
for anio, ruta in rasters.items():
    with rasterio.open(ruta) as fuente:
        datos = fuente.read(
            1,
            out_shape=(fuente.height // FACTOR_SUBMUESTREO, fuente.width // FACTOR_SUBMUESTREO),
            resampling=Resampling.nearest,
        )
    conteo = np.bincount(datos.ravel(), minlength=256)
    total = conteo[1:].sum()  # sin nodata (0)
    proporciones[anio] = {nombre: 100 * conteo[clase] / total for clase, nombre in CLASES_REPORTE.items()}

proporciones = pd.DataFrame(proporciones).T.rename_axis("anio")
print("% del territorio nacional:")
display(proporciones.round(2))

% del territorio nacional:


,Pastizal cultivado e inducido,Cultivo anual,Cultivo perenne,Mosaico de usos
anio,,,,
2002,5.34,17.27,0.15,0.0
2003,5.33,17.48,0.17,0.0
2004,5.25,17.33,0.18,0.0
2005,5.00,17.61,0.15,0.0
2006,5.16,18.77,0.16,0.0
2007,5.36,18.93,0.17,0.0
2008,5.72,19.06,0.17,0.0
2009,5.94,17.40,0.15,0.0
2010,5.82,16.52,0.14,0.0
